# Collatz Proof Assistant (Empirical)

This notebook starts an implementation-first workflow for computationally assisting Collatz conjecture research.

Scope of this notebook:
- deterministic computational experiments
- anomaly and invariant mining
- candidate lemma generation for later formalization

Important note: empirical verification is not a formal proof.

## 1. Environment Setup and Imports

Import required libraries, set deterministic behavior, and define notebook-wide utility settings.

In [1]:
from __future__ import annotations

import csv
import json
import random
import time
from dataclasses import asdict, dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Dict, Iterable, List, Tuple

try:
    import matplotlib.pyplot as plt
except Exception:
    plt = None

SEED = 20260318
random.seed(SEED)

def utc_now_iso() -> str:
    return datetime.now(timezone.utc).isoformat()

print(f"Notebook initialized at {utc_now_iso()} with seed={SEED}")

Notebook initialized at 2026-03-18T04:50:54.552770+00:00 with seed=20260318


## 2. Runtime Configuration

Adjust this cell only when changing the experiment setup.

In [2]:
@dataclass(frozen=True)
class RuntimeConfig:
    # Staged targets: 1e5 -> 1e6 -> 1e7
    start_n: int = 1
    end_n: int = 100_000
    iteration_cap: int = 20_000
    residue_modulus: int = 16
    top_k: int = 20
    output_root: str = "collatz_outputs"
    run_name: str = "stage_1e5"

CONFIG = RuntimeConfig()
RUN_DIR = Path(CONFIG.output_root) / CONFIG.run_name
RUN_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_JSON = RUN_DIR / "summary.json"
OUTPUT_CSV = RUN_DIR / "anomalies.csv"
LEMMA_JSON = RUN_DIR / "candidate_lemmas.json"
LEAN_FILE = RUN_DIR / "candidate_lemmas.lean"
COQ_FILE = RUN_DIR / "candidate_lemmas.v"

print("Run directory:", RUN_DIR.resolve())
print("Configured range:", CONFIG.start_n, "to", CONFIG.end_n)
print("To target 1e7, set end_n=10_000_000.")

Run directory: C:\Users\28576\Student_service_chatbot-2\collatz_outputs\stage_1e5
Configured range: 1 to 100000
To target 1e7, set end_n=10_000_000.


## 3. Input Data Stub and Sample Payloads

Use sample payloads for fast smoke tests before large range scans.

In [3]:
sample_payload = {
    "numbers": [1, 2, 3, 6, 7, 9, 27, 97, 871, 6171],
    "metadata": {
        "purpose": "smoke_test",
        "created_at_utc": utc_now_iso(),
    },
}

def build_range_payload(config: RuntimeConfig) -> Dict[str, Any]:
    return {
        "range": {"start": config.start_n, "end": config.end_n},
        "metadata": {
            "purpose": "range_scan",
            "created_at_utc": utc_now_iso(),
        },
    }

range_payload = build_range_payload(CONFIG)
print("Sample payload size:", len(sample_payload["numbers"]))
print("Range payload:", range_payload["range"])

Sample payload size: 10
Range payload: {'start': 1, 'end': 100000}


## 4. Core Function Implementation

Primary Collatz logic with explicit type-safe input validation and edge-case handling.

In [4]:
def validate_positive_int(value: Any, field_name: str) -> int:
    if isinstance(value, bool):
        raise TypeError(f"{field_name} must be an integer, got bool")
    if not isinstance(value, int):
        raise TypeError(f"{field_name} must be an integer, got {type(value).__name__}")
    if value < 1:
        raise ValueError(f"{field_name} must be >= 1, got {value}")
    return value


def factor_out_twos(x: int) -> Tuple[int, int]:
    x = validate_positive_int(x, "x")
    shift = (x & -x).bit_length() - 1
    return x >> shift, shift


def collatz_step(n: int) -> int:
    n = validate_positive_int(n, "n")
    return n // 2 if n % 2 == 0 else 3 * n + 1


def collatz_metrics(n: int, iteration_cap: int) -> Dict[str, Any]:
    n = validate_positive_int(n, "n")
    iteration_cap = validate_positive_int(iteration_cap, "iteration_cap")

    current = n
    steps = 0
    odd_steps = 0
    even_steps = 0
    max_value = n

    while current != 1 and steps < iteration_cap:
        if current & 1:
            current = 3 * current + 1
            steps += 1
            odd_steps += 1
            if current > max_value:
                max_value = current

        reduced, shift = factor_out_twos(current)
        if shift:
            current = reduced
            steps += shift
            even_steps += shift

    terminated = current == 1
    return {
        "n": n,
        "terminated": terminated,
        "steps": steps,
        "max_value": max_value,
        "peak_ratio": max_value / n,
        "odd_steps": odd_steps,
        "even_steps": even_steps,
        "hit_cap": not terminated,
    }


def collatz_trace(n: int, iteration_cap: int, max_terms: int = 300) -> List[int]:
    n = validate_positive_int(n, "n")
    iteration_cap = validate_positive_int(iteration_cap, "iteration_cap")
    max_terms = validate_positive_int(max_terms, "max_terms")

    seq = [n]
    current = n
    steps = 0
    while current != 1 and steps < iteration_cap and len(seq) < max_terms:
        current = collatz_step(current)
        seq.append(current)
        steps += 1
    return seq

## 5. Pipeline Orchestration Function

Compose core helpers into one end-to-end workflow for scanning, summary building, anomaly extraction, and export scaffolding.

In [5]:
def iter_payload_numbers(payload: Dict[str, Any], config: RuntimeConfig) -> Iterable[int]:
    if "numbers" in payload:
        for idx, value in enumerate(payload["numbers"]):
            yield validate_positive_int(value, f"numbers[{idx}]")
        return

    range_info = payload.get("range", {})
    start = validate_positive_int(range_info.get("start", config.start_n), "range.start")
    end = validate_positive_int(range_info.get("end", config.end_n), "range.end")
    if end < start:
        raise ValueError("range.end must be >= range.start")

    for n in range(start, end + 1):
        yield n


def init_summary(config: RuntimeConfig) -> Dict[str, Any]:
    residue_stats = {
        str(r): {
            "count": 0,
            "terminated": 0,
            "steps_sum": 0,
            "peak_ratio_sum": 0.0,
            "max_steps": 0,
            "max_peak_ratio": 0.0,
        }
        for r in range(config.residue_modulus)
    }

    return {
        "config": asdict(config),
        "started_at_utc": utc_now_iso(),
        "completed_at_utc": None,
        "count": 0,
        "terminated_count": 0,
        "hit_cap_count": 0,
        "steps_sum": 0,
        "odd_steps_sum": 0,
        "even_steps_sum": 0,
        "min_n": None,
        "max_n": None,
        "max_steps": {"n": 1, "steps": 0},
        "max_peak": {"n": 1, "peak_ratio": 1.0, "max_value": 1, "steps": 0},
        "steps_histogram": {},
        "residue_stats": residue_stats,
        "top_steps": [],
        "top_peak": [],
        "non_terminated_examples": [],
        "sample_rows": [],
        "elapsed_seconds": 0.0,
        "throughput_n_per_sec": 0.0,
        "avg_steps": 0.0,
        "termination_rate": 0.0,
        "scan_range": {"start": None, "end": None},
        "residue_table": [],
    }


def keep_top_k(rows: List[Dict[str, Any]], row: Dict[str, Any], key: str, k: int) -> None:
    rows.append(row)
    rows.sort(key=lambda item: (-item[key], item["n"]))
    del rows[k:]


def update_summary(summary: Dict[str, Any], metrics: Dict[str, Any], config: RuntimeConfig) -> None:
    n = metrics["n"]
    summary["count"] += 1
    summary["steps_sum"] += metrics["steps"]
    summary["odd_steps_sum"] += metrics["odd_steps"]
    summary["even_steps_sum"] += metrics["even_steps"]

    if summary["min_n"] is None or n < summary["min_n"]:
        summary["min_n"] = n
    if summary["max_n"] is None or n > summary["max_n"]:
        summary["max_n"] = n

    if metrics["terminated"]:
        summary["terminated_count"] += 1
    else:
        summary["hit_cap_count"] += 1
        if len(summary["non_terminated_examples"]) < config.top_k:
            summary["non_terminated_examples"].append(
                {
                    "n": n,
                    "steps": metrics["steps"],
                    "max_value": metrics["max_value"],
                    "peak_ratio": metrics["peak_ratio"],
                }
            )

    if metrics["steps"] > summary["max_steps"]["steps"]:
        summary["max_steps"] = {"n": n, "steps": metrics["steps"]}

    if metrics["peak_ratio"] > summary["max_peak"]["peak_ratio"]:
        summary["max_peak"] = {
            "n": n,
            "peak_ratio": metrics["peak_ratio"],
            "max_value": metrics["max_value"],
            "steps": metrics["steps"],
        }

    step_key = str(metrics["steps"]) if metrics["terminated"] else f"cap:{config.iteration_cap}"
    summary["steps_histogram"][step_key] = summary["steps_histogram"].get(step_key, 0) + 1

    residue_key = str(n % config.residue_modulus)
    residue = summary["residue_stats"][residue_key]
    residue["count"] += 1
    residue["terminated"] += 1 if metrics["terminated"] else 0
    residue["steps_sum"] += metrics["steps"]
    residue["peak_ratio_sum"] += metrics["peak_ratio"]
    residue["max_steps"] = max(residue["max_steps"], metrics["steps"])
    residue["max_peak_ratio"] = max(residue["max_peak_ratio"], metrics["peak_ratio"])

    keep_top_k(
        summary["top_steps"],
        {
            "n": n,
            "steps": metrics["steps"],
            "max_value": metrics["max_value"],
            "peak_ratio": metrics["peak_ratio"],
        },
        key="steps",
        k=config.top_k,
    )

    keep_top_k(
        summary["top_peak"],
        {
            "n": n,
            "steps": metrics["steps"],
            "max_value": metrics["max_value"],
            "peak_ratio": metrics["peak_ratio"],
        },
        key="peak_ratio",
        k=config.top_k,
    )

    if len(summary["sample_rows"]) < 10:
        summary["sample_rows"].append(metrics)


def finalize_summary(summary: Dict[str, Any], elapsed_seconds: float) -> None:
    count = summary["count"]
    summary["elapsed_seconds"] = elapsed_seconds
    summary["throughput_n_per_sec"] = count / elapsed_seconds if elapsed_seconds > 0 else 0.0
    summary["avg_steps"] = summary["steps_sum"] / count if count else 0.0
    summary["termination_rate"] = summary["terminated_count"] / count if count else 0.0
    summary["scan_range"] = {"start": summary["min_n"], "end": summary["max_n"]}

    residue_table = []
    for residue_key, stats in sorted(summary["residue_stats"].items(), key=lambda item: int(item[0])):
        c = stats["count"]
        residue_table.append(
            {
                "residue": int(residue_key),
                "count": c,
                "termination_rate": stats["terminated"] / c if c else 0.0,
                "avg_steps": stats["steps_sum"] / c if c else 0.0,
                "avg_peak_ratio": stats["peak_ratio_sum"] / c if c else 0.0,
                "max_steps": stats["max_steps"],
                "max_peak_ratio": stats["max_peak_ratio"],
            }
        )

    summary["residue_table"] = residue_table
    summary["completed_at_utc"] = utc_now_iso()


def run_pipeline(config: RuntimeConfig, payload: Dict[str, Any], progress_every: int = 50_000) -> Dict[str, Any]:
    start = time.perf_counter()
    summary = init_summary(config)

    for index, n in enumerate(iter_payload_numbers(payload, config), start=1):
        metrics = collatz_metrics(n, config.iteration_cap)
        update_summary(summary, metrics, config)

        if progress_every and index % progress_every == 0:
            elapsed = time.perf_counter() - start
            speed = index / elapsed if elapsed > 0 else 0.0
            print(f"Processed {index:,} numbers at {speed:,.0f} n/s")

    finalize_summary(summary, time.perf_counter() - start)
    return summary


def build_candidate_lemmas(summary: Dict[str, Any]) -> List[Dict[str, Any]]:
    cards: List[Dict[str, Any]] = []
    scan_start = summary["scan_range"]["start"]
    scan_end = summary["scan_range"]["end"]
    hit_cap = summary["hit_cap_count"]

    cards.append(
        {
            "id": "empirical_termination_on_scanned_range",
            "statement": f"All scanned n in [{scan_start}, {scan_end}] reached 1 within the configured iteration cap.",
            "status": "supported" if hit_cap == 0 else "inconclusive",
            "evidence": {
                "count": summary["count"],
                "hit_cap_count": hit_cap,
                "iteration_cap": summary["config"]["iteration_cap"],
            },
        }
    )

    cards.append(
        {
            "id": "empirical_upper_bound_on_stopping_time",
            "statement": f"For scanned n, stopping time did not exceed {summary['max_steps']['steps']}.",
            "status": "supported",
            "evidence": summary["max_steps"],
        }
    )

    residue_table = summary.get("residue_table", [])
    if residue_table:
        fastest = min(residue_table, key=lambda row: row["avg_steps"])
        slowest = max(residue_table, key=lambda row: row["avg_steps"])
        cards.append(
            {
                "id": "empirical_residue_class_variation",
                "statement": "Average stopping times differ by residue class modulo the configured modulus.",
                "status": "supported",
                "evidence": {
                    "modulus": summary["config"]["residue_modulus"],
                    "fastest_avg_steps": fastest,
                    "slowest_avg_steps": slowest,
                },
            }
        )

    return cards


def export_formal_templates(summary: Dict[str, Any], lemmas: List[Dict[str, Any]], lean_path: Path, coq_path: Path) -> None:
    scan_start = summary["scan_range"]["start"]
    scan_end = summary["scan_range"]["end"]
    cap = summary["config"]["iteration_cap"]

    lean_lines = [
        "-- Auto-generated placeholders from empirical Collatz scan.",
        "-- This file is a scaffold for future formalization.",
        f"-- Tested range: [{scan_start}, {scan_end}], cap={cap}",
        "",
        f"theorem collatz_termination_upto_{scan_end} : Prop := by",
        "  -- TODO: replace Prop with a precise formal statement.",
        "  sorry",
        "",
        "-- Candidate lemmas from empirical evidence:",
    ]
    for card in lemmas:
        lean_lines.append(f"-- {card['id']}: {card['statement']} [{card['status']}]")
    lean_path.write_text("\n".join(lean_lines) + "\n", encoding="utf-8")

    coq_lines = [
        "(* Auto-generated placeholders from empirical Collatz scan. *)",
        "(* This file is a scaffold for future formalization. *)",
        f"(* Tested range: [{scan_start}, {scan_end}], cap={cap} *)",
        "",
        f"Theorem collatz_termination_upto_{scan_end} : True.",
        "Proof.",
        "  (* TODO: replace True with a precise formal statement. *)",
        "Admitted.",
        "",
        "(* Candidate lemmas from empirical evidence: *)",
    ]
    for card in lemmas:
        coq_lines.append(f"(* {card['id']}: {card['statement']} [{card['status']}] *)")
    coq_path.write_text("\n".join(coq_lines) + "\n", encoding="utf-8")


def write_outputs(
    summary: Dict[str, Any],
    output_json: Path,
    output_csv: Path,
    lemma_json: Path,
    lean_file: Path,
    coq_file: Path,
) -> List[Dict[str, Any]]:
    output_json.write_text(json.dumps(summary, indent=2), encoding="utf-8")

    with output_csv.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(
            f,
            fieldnames=["kind", "rank", "n", "steps", "max_value", "peak_ratio", "trace_prefix"],
        )
        writer.writeheader()

        for rank, row in enumerate(summary["top_steps"], start=1):
            trace_prefix = collatz_trace(row["n"], summary["config"]["iteration_cap"], max_terms=20)
            writer.writerow(
                {
                    "kind": "top_steps",
                    "rank": rank,
                    "n": row["n"],
                    "steps": row["steps"],
                    "max_value": row["max_value"],
                    "peak_ratio": row["peak_ratio"],
                    "trace_prefix": " ".join(str(x) for x in trace_prefix),
                }
            )

        for rank, row in enumerate(summary["top_peak"], start=1):
            trace_prefix = collatz_trace(row["n"], summary["config"]["iteration_cap"], max_terms=20)
            writer.writerow(
                {
                    "kind": "top_peak",
                    "rank": rank,
                    "n": row["n"],
                    "steps": row["steps"],
                    "max_value": row["max_value"],
                    "peak_ratio": row["peak_ratio"],
                    "trace_prefix": " ".join(str(x) for x in trace_prefix),
                }
            )

        for rank, row in enumerate(summary["non_terminated_examples"], start=1):
            writer.writerow(
                {
                    "kind": "hit_cap",
                    "rank": rank,
                    "n": row["n"],
                    "steps": row["steps"],
                    "max_value": row["max_value"],
                    "peak_ratio": row["peak_ratio"],
                    "trace_prefix": "",
                }
            )

    lemmas = build_candidate_lemmas(summary)
    lemma_json.write_text(
        json.dumps({"generated_at_utc": utc_now_iso(), "lemmas": lemmas}, indent=2),
        encoding="utf-8",
    )
    export_formal_templates(summary, lemmas, lean_file, coq_file)

    return lemmas

### 5.1 Checkpointed Range Scanner (Resume-Safe)

Use this scanner for large ranges. It writes progress to checkpoint files and supports resume after interruption.

In [8]:
CHECKPOINT_FILE = RUN_DIR / "checkpoint.json"

def save_checkpoint(path: Path, payload: Dict[str, Any]) -> None:
    temp_path = path.with_suffix(path.suffix + ".tmp")
    temp_path.write_text(json.dumps(payload, indent=2), encoding="utf-8")
    temp_path.replace(path)


def load_checkpoint(path: Path, config: RuntimeConfig) -> Dict[str, Any] | None:
    if not path.exists():
        return None

    payload = json.loads(path.read_text(encoding="utf-8"))
    checkpoint_config = payload.get("config", {})
    expected = asdict(config)

    # Require full config equality for deterministic resume behavior.
    if checkpoint_config != expected:
        raise ValueError(
            "Checkpoint config mismatch. "
            "Delete checkpoint or reuse exactly the same RuntimeConfig."
        )

    return payload


def run_checkpointed_range_pipeline(
    config: RuntimeConfig,
    checkpoint_path: Path,
    chunk_size: int = 50_000,
    checkpoint_every_chunks: int = 1,
    resume: bool = True,
) -> Dict[str, Any]:
    chunk_size = validate_positive_int(chunk_size, "chunk_size")
    checkpoint_every_chunks = validate_positive_int(
        checkpoint_every_chunks, "checkpoint_every_chunks"
    )
    if config.end_n < config.start_n:
        raise ValueError("end_n must be >= start_n")

    wall_start = time.perf_counter()
    checkpoint_data = load_checkpoint(checkpoint_path, config) if resume else None

    if checkpoint_data:
        summary = checkpoint_data["summary"]
        next_n = int(checkpoint_data["next_n"])
        chunk_index = int(checkpoint_data.get("chunk_index", 0))
        elapsed_before_resume = float(checkpoint_data.get("elapsed_seconds", 0.0))
        print(f"Resuming from checkpoint at n={next_n:,}")
    else:
        summary = init_summary(config)
        next_n = config.start_n
        chunk_index = 0
        elapsed_before_resume = 0.0
        print("Starting new checkpointed range scan")

    total_target = config.end_n - config.start_n + 1

    while next_n <= config.end_n:
        chunk_end = min(next_n + chunk_size - 1, config.end_n)
        for n in range(next_n, chunk_end + 1):
            metrics = collatz_metrics(n, config.iteration_cap)
            update_summary(summary, metrics, config)

        chunk_index += 1
        elapsed_now = elapsed_before_resume + (time.perf_counter() - wall_start)

        if chunk_index % checkpoint_every_chunks == 0 or chunk_end == config.end_n:
            save_checkpoint(
                checkpoint_path,
                {
                    "config": asdict(config),
                    "summary": summary,
                    "next_n": chunk_end + 1,
                    "chunk_index": chunk_index,
                    "elapsed_seconds": elapsed_now,
                    "updated_at_utc": utc_now_iso(),
                },
            )

            processed = summary["count"]
            progress = (processed / total_target) * 100.0 if total_target else 100.0
            speed = processed / elapsed_now if elapsed_now > 0 else 0.0
            print(
                f"chunk={chunk_index:,} n={chunk_end:,} "
                f"done={processed:,}/{total_target:,} ({progress:.2f}%) "
                f"speed={speed:,.0f} n/s"
            )

        next_n = chunk_end + 1

    total_elapsed = elapsed_before_resume + (time.perf_counter() - wall_start)
    finalize_summary(summary, total_elapsed)
    return summary

print("Checkpoint file:", CHECKPOINT_FILE)

Checkpoint file: collatz_outputs\stage_1e5\checkpoint.json


## 6. Smoke Test Execution

Run the full pipeline on sample input and inspect key checkpoints.

In [6]:
smoke_summary = run_pipeline(CONFIG, sample_payload, progress_every=0)

print(
    json.dumps(
        {
            "count": smoke_summary["count"],
            "termination_rate": smoke_summary["termination_rate"],
            "max_steps": smoke_summary["max_steps"],
            "max_peak": smoke_summary["max_peak"],
            "elapsed_seconds": smoke_summary["elapsed_seconds"],
        },
        indent=2,
    )
)

print("Top step outliers (sample run):")
for row in smoke_summary["top_steps"][:5]:
    print(row)

{
  "count": 10,
  "termination_rate": 1.0,
  "max_steps": {
    "n": 6171,
    "steps": 261
  },
  "max_peak": {
    "n": 27,
    "peak_ratio": 341.9259259259259,
    "max_value": 9232,
    "steps": 111
  },
  "elapsed_seconds": 0.0001664000010350719
}
Top step outliers (sample run):
{'n': 6171, 'steps': 261, 'max_value': 975400, 'peak_ratio': 158.0619024469292}
{'n': 871, 'steps': 178, 'max_value': 190996, 'peak_ratio': 219.28358208955223}
{'n': 97, 'steps': 118, 'max_value': 9232, 'peak_ratio': 95.17525773195877}
{'n': 27, 'steps': 111, 'max_value': 9232, 'peak_ratio': 341.9259259259259}
{'n': 9, 'steps': 19, 'max_value': 52, 'peak_ratio': 5.777777777777778}


## 7. Assertions and Output Persistence

Validate known values, then persist outputs for downstream analysis and formalization workflows.

In [9]:
KNOWN_STOPPING_TIMES = {
    1: 0,
    2: 1,
    3: 7,
    6: 8,
    7: 16,
    9: 19,
    27: 111,
}

for n, expected_steps in KNOWN_STOPPING_TIMES.items():
    observed_steps = collatz_metrics(n, iteration_cap=20_000)["steps"]
    assert observed_steps == expected_steps, (
        f"Stopping time mismatch for n={n}: expected {expected_steps}, got {observed_steps}"
    )

# Keep smoke outputs separate from large staged scans.
smoke_output_json = RUN_DIR / "smoke_summary.json"
smoke_output_csv = RUN_DIR / "smoke_anomalies.csv"
smoke_lemma_json = RUN_DIR / "smoke_candidate_lemmas.json"
smoke_lean_file = RUN_DIR / "smoke_candidate_lemmas.lean"
smoke_coq_file = RUN_DIR / "smoke_candidate_lemmas.v"

lemmas = write_outputs(
    smoke_summary,
    output_json=smoke_output_json,
    output_csv=smoke_output_csv,
    lemma_json=smoke_lemma_json,
    lean_file=smoke_lean_file,
    coq_file=smoke_coq_file,
)

print("Assertions passed and smoke outputs written:")
print("-", smoke_output_json)
print("-", smoke_output_csv)
print("-", smoke_lemma_json)
print("-", smoke_lean_file)
print("-", smoke_coq_file)
print("Generated candidate lemmas:", len(lemmas))

# Optional staged range run with checkpoint/resume (1e5 -> 1e6 -> 1e7)
# large_summary = run_checkpointed_range_pipeline(
#     CONFIG,
#     checkpoint_path=CHECKPOINT_FILE,
#     chunk_size=50_000,
#     checkpoint_every_chunks=1,
#     resume=True,
# )
# large_lemmas = write_outputs(
#     large_summary,
#     output_json=OUTPUT_JSON,
#     output_csv=OUTPUT_CSV,
#     lemma_json=LEMMA_JSON,
#     lean_file=LEAN_FILE,
#     coq_file=COQ_FILE,
# )

Assertions passed and smoke outputs written:
- collatz_outputs\stage_1e5\smoke_summary.json
- collatz_outputs\stage_1e5\smoke_anomalies.csv
- collatz_outputs\stage_1e5\smoke_candidate_lemmas.json
- collatz_outputs\stage_1e5\smoke_candidate_lemmas.lean
- collatz_outputs\stage_1e5\smoke_candidate_lemmas.v
Generated candidate lemmas: 3
